In [1]:
import numpy as np
import pandas as pd
import snowflake.connector
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

conn = snowflake.connector.connect(connection_name="criteo_svc")
s = conn.cursor().execute("SELECT * FROM lift_summary").fetch_pandas_all()

t = s[s.TREATMENT == 1].iloc[0]
c = s[s.TREATMENT == 0].iloc[0]
n_t, n_c = int(t["N"]), int(c["N"])
ratio = n_c / n_t          # control size relative to treatment (about 0.176)

power = NormalIndPower()
ALPHA, POWER = 0.05, 0.80

In [2]:
def rate_from_h(p_c, h):
    # convert an effect size (Cohen's h) back into a treatment rate
    return np.sin(np.arcsin(np.sqrt(p_c)) + h / 2) ** 2

rows = []
for metric in ["VISITS", "CONVERSIONS"]:
    p_c = c[metric] / n_c
    p_t = t[metric] / n_t

    # 1. Smallest lift this experiment could detect
    h_mde = power.solve_power(nobs1=n_t, ratio=ratio, alpha=ALPHA, power=POWER)
    mde = rate_from_h(p_c, h_mde) - p_c

    # 2. Users needed to detect the lift we actually observed (same 85/15 split)
    h_obs = proportion_effectsize(p_t, p_c)
    n_t_needed = power.solve_power(effect_size=h_obs, ratio=ratio, alpha=ALPHA, power=POWER)
    n_total_needed = n_t_needed * (1 + ratio)

    rows.append({
        "metric": metric.lower(),
        "cohens_h": h_obs,
        "observed_abs_lift": p_t - p_c,
        "mde_abs": mde,
        "lift_vs_mde": (p_t - p_c) / mde,
        "users_needed_85_15": n_total_needed,
        "users_actual": n_t + n_c,
    })

power_results = pd.DataFrame(rows)
power_results.T

,0,1
metric,visits,conversions
cohens_h,0.050863,0.023159
observed_abs_lift,0.010342,0.001152
mde_abs,0.000403,0.000093
lift_vs_mde,25.648034,12.33596
users_needed_85_15,23795.771353,114781.332332
users_actual,13979592,13979592


In [3]:
curve = []
p_c_visit = c["VISITS"] / n_c
for rel_lift in np.arange(0.01, 0.31, 0.01):        # 1% to 30% relative lift
    p_t_hyp = p_c_visit * (1 + rel_lift)
    h = proportion_effectsize(p_t_hyp, p_c_visit)
    n_t_req = power.solve_power(effect_size=h, ratio=ratio, alpha=ALPHA, power=POWER)
    curve.append({
        "relative_lift": rel_lift,
        "users_needed": n_t_req * (1 + ratio),
    })

curve = pd.DataFrame(curve)
curve.to_csv("../data/power_curve.csv", index=False)
power_results.to_csv("../data/power_results.csv", index=False)
curve.head(10)

,relative_lift,users_needed
0,0.01,1.557342e+07
1,0.02,3.911878e+06
2,0.03,1.746821e+06
3,0.04,9.871910e+05
4,0.05,6.347406e+05
5,0.06,4.428269e+05
6,0.07,3.268330e+05
7,0.08,2.513698e+05
8,0.09,1.995101e+05
9,0.10,1.623278e+05
